# NBA watchability scoring

The NBA counterpart of `scoring.ipynb`: compute the signals from ESPN play-by-play and win probability, score games against a 2023-24 to 2025-26 reference distribution, and check the ranking against your own judgment.

All logic lives in `nba_watchability.py`; this notebook only runs and inspects it. Seasons are named by their start year (2025 = 2025-26). The first run downloads about 4,000 game summaries into `data/raw/espn_nba/` (run `python -P nba_watchability.py` to backfill from a terminal). **Spoiler note:** past seasons show scores; the current-season view shows only matchup, tier and stars.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import nba_watchability as n

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

REFERENCE_SEASONS = [2023, 2024, 2025]
CURRENT_SEASON = 2026

games = n.load_schedule(REFERENCE_SEASONS + [CURRENT_SEASON])   # refresh=True re-downloads cached days
final = games[games.completed]
summaries = n.load_summaries(final.game_id.tolist())
signals = n.build_signals(games, summaries)
reference = signals[signals.season.isin(REFERENCE_SEASONS)]
scored = n.score(signals, reference)
ref_scored = scored[scored.season.isin(REFERENCE_SEASONS)]
print(f"{len(reference)} reference games, {len(scored) - len(reference)} current-season games, "
      f"{len(final) - len(signals)} final games without win probability")

## Raw signal distributions (reference seasons)

In [ ]:
reference[n.SIGNALS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 5.5))
for ax, s in zip(axes.flat, n.SIGNALS):
    ax.hist(reference[s], bins=30, color="#4C78A8")
    ax.set_title(f"{s} (w={n.WEIGHTS[s]})", fontsize=10)
ax = axes.flat[-1]
ax.hist(ref_scored.score, bins=30, color="#F58518")
ax.set_title("final score", fontsize=10)
fig.tight_layout()

## Are the signals redundant?

High correlation means two signals double-count the same thing. Excitement, closeness and comeback are expected to overlap somewhat. `lead_changes` is computed but unweighted; it's here to see whether it adds anything.

In [ ]:
reference[n.SIGNALS + ["lead_changes"]].corr(method="spearman").round(2)

## Tier calibration

The target is 2-3 **Must watch** games per regular-season week (Monday to Sunday). This shows how the current cutoffs land and what cutoffs would hit the target.

In [ ]:
reg = ref_scored[ref_scored.season_type == "REG"].assign(week=lambda d: n.week_of(d.night))
full_weeks = reg.groupby(["season", "week"]).filter(lambda w: len(w) >= 30)
per_week = full_weeks.groupby(["season", "week"]).tier.apply(lambda t: (t == "Must watch").sum())
per_night = reg.groupby("night").tier.apply(lambda t: (t == "Worth it").sum())
print("Tier share:", ref_scored.tier.value_counts(normalize=True).round(3).to_dict())
print(f"Must watch per week at cutoff {n.TIERS[0][0]}: mean {per_week.mean():.2f}, median {per_week.median():.0f}")
print(f"Worth it per night at cutoff {n.TIERS[1][0]}: mean {per_night.mean():.2f}")

games_per_week = full_weeks.groupby(["season", "week"]).size().mean()
for target in (2, 2.5, 3):
    cutoff = full_weeks.score.quantile(1 - target / games_per_week)
    print(f"  {target} Must watch per week  ->  cutoff {cutoff:.1f}")

## Best and worst reference games

In [ ]:
cols = ["game_id", "night", "away_team", "away_score", "home_team", "home_score", "score", "tier",
        "excitement", "late_closeness", "comeback", "highlight_plays", "scoring", "star", "stakes", "big_moment", "notes"]
ref_scored.nlargest(15, "score")[cols]

In [ ]:
ref_scored.nsmallest(10, "score")[cols]

## Why it ranked here: points per signal for the top reference game

In [ ]:
top = ref_scored.loc[ref_scored.score.idxmax()]
print(top.game_id, top.away_team, "@", top.home_team, top.score)
n.contributions(top)

## Current season, spoiler safe

Matchup, tier and stars only, best first, for one night. Defaults to the latest night with games.

In [ ]:
current = scored[scored.season == CURRENT_SEASON]
NIGHT = current.night.max() if len(current) else None   # set "2026-10-20" to override
night = current[current.night == NIGHT].sort_values("score", ascending=False)
night.assign(matchup=night.away_team + " @ " + night.home_team)[
    ["matchup", "tier", "stars", "national", "playoff", "play_in", "cup_knockout"]
].reset_index(drop=True)

## Blind check: does the model agree with you?

1. Run the next cell. It lists one night's games in random order with no scores: the latest night of the current season with at least 6 games, or of 2025-26 until the new season has one. Set `CHECK_NIGHT` for any other night.
2. Put the game_ids in `my_ranking`, best first.
3. Run the cell after it to reveal the model's ranking. The bar: at least 2 of your top 3 in the model's top 3.

In [ ]:
sizes = scored.groupby("night").size()
CHECK_NIGHT = sizes[sizes >= 6].index.max()   # latest full night; set "2026-10-21" to override

check = scored[scored.night == CHECK_NIGHT]
print(f"{CHECK_NIGHT}: {len(check)} games")
check.sample(frac=1, random_state=0).assign(matchup=lambda d: d.away_team + " @ " + d.home_team)[
    ["game_id", "matchup"]
].reset_index(drop=True)

In [ ]:
my_ranking = []

if len(my_ranking) >= 3:
    print(f"Overlap in top 3: {n.top_k_overlap(my_ranking, check, k=3)} of 3")
    display(check.sort_values("score", ascending=False)[cols])
else:
    print("Fill in my_ranking (at least 3 game_ids) to run the check.")

## Save scores for the ingestion job to compare against

In [ ]:
out = n.DATA_DIR.parent.parent / "scored_nba_games.csv"
scored.drop(columns=[c for c in scored.columns if c.startswith("pct_")]).to_csv(out, index=False)
print(f"Wrote {len(scored)} games to {out}")